# gpx-to-geojson demo

Convert the Cairo landmarks GPX (waypoints and a route) to GeoJSON. The first cell sets things up: in a fresh runtime such as Colab it installs the package and downloads `examples/cairo.gpx`; in a repository checkout with the package installed it does nothing.

In [ ]:
import importlib.util
import os
import subprocess
import sys
import urllib.request

REPO = "https://github.com/AmmarYasser455/gpx-to-geojson"
if importlib.util.find_spec("gpx_to_geojson") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+" + REPO])
if not os.path.exists("examples/cairo.gpx"):
    os.makedirs("examples", exist_ok=True)
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/AmmarYasser455/gpx-to-geojson/main/examples/cairo.gpx",
        "examples/cairo.gpx",
    )

In [ ]:
from gpx_to_geojson import convert

fc = convert("examples/cairo.gpx")
print(len(fc["features"]), "features")

In [ ]:
for f in fc["features"]:
    p = f["properties"]
    print(p["kind"], p["name"], f["geometry"]["type"], f["geometry"]["coordinates"] if p["kind"] == "waypoint" else "")

In [ ]:
route = next(f for f in fc["features"] if f["properties"]["kind"] == "route")
print("geodesic length: %.2f km" % (route["properties"]["distance_m"] / 1000))
assert 14 < route["properties"]["distance_m"] / 1000 < 17.5
# Route-point names are kept as a per-vertex array, aligned with the coordinates.
for name, (lon, lat) in zip(route["properties"]["point_name"], route["geometry"]["coordinates"]):
    print(f"{name}: lon={lon}, lat={lat}")

In [ ]:
from gpx_to_geojson import parse_gpx

gpx = '''<gpx version="1.1"><trk><name>demo</name><desc>synthetic</desc><trkseg>
<trkpt lat="0" lon="0"><ele>10</ele><time>2024-01-01T10:00:00Z</time></trkpt>
<trkpt lat="0" lon="1"><ele>30</ele><time>2024-01-01T10:10:00Z</time></trkpt>
</trkseg></trk></gpx>'''
track = parse_gpx(gpx)["features"][0]["properties"]
print(track["desc"], track["distance_m"], track["duration_s"], track["elevation_max_m"])